# Practical Experiment 11: Transfer Learning and Fine-Tuning Pretrained Models

---
### **Aim / Objective:**
To leverage pretrained deep architectures (e.g., VGG16, ResNet50) on custom tasks by freezing base layers, training a custom classification head, and performing fine-tuning.

---
### **Theory & Overview:**
Transfer Learning reuses features learned from massive datasets (like ImageNet with 1.4M images):
1. **Feature Extraction**: Base convolutional layers are frozen (`trainable = False`). Only the custom top classifier is trained.
2. **Fine-Tuning**: Select top convolutional layers are unfrozen and trained jointly with a very small learning rate ($10^{-5}$) to adapt higher-level filters to specific domains.



### **1. Loading Pretrained VGG16 / MobileNetV2 Base**


In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
from keras import layers
import numpy as np

# Offline fallback: MobileNetV2 architecture without downloading ImageNet weights.
base_model = keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights=None
)
base_model.trainable = False
print("MobileNetV2 base created without ImageNet download (offline mode).")
print(f"Trainable variables in frozen base: {len(base_model.trainable_variables)}")

MobileNetV2 base created without ImageNet download (offline mode).
Trainable variables in frozen base: 0


### **2. Attaching Custom Classification Head**


In [2]:
inputs = keras.Input(shape=(96, 96, 3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,264,389 (8.64 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

### **3. Phase 1: Training the Classification Head**


In [3]:
rng = np.random.default_rng(11)
X_sample = rng.integers(0, 256, (4, 96, 96, 3)).astype("float32")
y_sample = rng.integers(0, 5, (4,))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Phase 1: Training classification head with frozen base:")
model.fit(X_sample, y_sample, epochs=1, batch_size=4, verbose=1)

Phase 1: Training classification head with frozen base:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/step - accuracy: 0.0000e+00 - loss: 1.6094


### **4. Phase 2: Unfreezing Top Layers for Fine-Tuning**


In [4]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

print(f"Trainable variables after unfreezing top layers: {len(base_model.trainable_variables)}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Phase 2: Fine-tuning top 20 convolutional layers:")
model.fit(X_sample, y_sample, epochs=1, batch_size=4, verbose=1)
print("Fine-tuning phase completed successfully.")

Trainable variables after unfreezing top layers: 21
Phase 2: Fine-tuning top 20 convolutional layers:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - accuracy: 0.2500 - loss: 1.6086
Fine-tuning phase completed successfully.


---
### **Conclusion & Learning Outcomes:**
Mastered the 2-stage transfer learning paradigm: freezing base representations for feature extraction followed by selective layer unfreezing with low learning rates for domain-specific fine-tuning.



## Output
The code cells above are ready to execute in Jupyter Notebook. Run the cells from top to bottom to generate the actual model summaries, training logs, accuracy/loss values, and plots.


**Execution note:** The original experiment requests ImageNet pretrained weights. The execution environment was offline, so `weights=None` was used to avoid an external download. The two-stage frozen-base and fine-tuning workflow is preserved and the notebook contains actual execution outputs.